# Structure Prediction

This notebook demonstrates the `predict_structures` dispatch function, which provides a unified entry point for running any structure prediction tool in proto-tools. Rather than importing a specific tool like ESMFold or Protenix directly, you pass the tool name as a string and `predict_structures` handles input construction, dispatch, and output normalization. This pattern is particularly useful when the choice of backend is determined at runtime or configured by a user. The example below predicts a protein-ligand complex — RuvB (a bacterial helicase) bound to ADP — using Boltz2.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("boltz2")
display_overview("boltz2")
display_docs_section("boltz2", "Background")

## Available tools

In [ ]:
# Show available functions for the structure prediction tools demonstrated in this notebook
display_available_tools("boltz2")

### `predict_structures`

The `predict_structures` dispatch function accepts one or more `StructurePredictionComplex` objects and a `toolkit` string, then routes the request to the appropriate tool implementation. This makes it straightforward to swap backends — ESMFold for speed, Boltz2 or Protenix for multi-modal accuracy — without changing any input construction code. Each complex is a list of chains with sequences and entity types. Protein chains use amino acid sequences, while ligands use SMILES notation.

In [ ]:
from proto_tools.tools.structure_prediction.dispatch import predict_structures
from proto_tools.tools.structure_prediction.shared_data_models import StructurePredictionComplex

In [ ]:
# Display input API reference for Boltz2 (the tool used in this example)
display_api_reference("boltz2", "input", "run_boltz2")

# RuvB helicase sequence (bacterial, from Thermus thermophilus)
RuvB = "VERTLRPQYFKEYIGQDKVKDQLKIFIEAAKLRDEALDHTLLFGPPGLGKTTMAFVIANEMGVNLKQTSGPAIEKAGDLVAILNDLEPGDILFIDEIHRMPMAVEEVLYSAMEDYYIDIMIGAGETSRSVHLDLPPFTLVGATTRAGMLSNPLRARFGINGHMEYYELPDLTEIVERTSEIFEMTITPEAALELARRSRGTPRIANRLLKRVRDYAQIMGDGVIDDKIADQALTMLDVDHEGLDYVDQKILRTMIEMYGGGPVGLGTLSVNIAEERETVEDMYEPYLIQKGFIMRTRTGRVATAKAYEHMGYDYTRDN"

# ADP (adenosine diphosphate) SMILES
ADP = "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@](=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]1O"

# Create protein-ligand complex
input_complex = StructurePredictionComplex(
    chains=[
        {"sequence": RuvB, "entity_type": "protein"},
        {"sequence": ADP, "entity_type": "ligand"},
    ]
)

In [ ]:
# Display config API reference for Boltz2
display_api_reference("boltz2", "config", "run_boltz2")

# Tool config can be passed as a dict; predict_structures will instantiate the correct config class
tool_config = {"verbose": True}

In [ ]:
# Run the prediction using the dispatch interface
output = predict_structures(
    complexes=input_complex,
    toolkit="boltz2",
    tool_config=tool_config,
)

In [ ]:
# Display output API reference for Boltz2
display_api_reference("boltz2", "output", "run_boltz2")

structure = output.structures[0]

# Print basic structure metrics
print(f"Chain IDs:          {structure.get_chain_ids()}")
print(f"Number of residues: {structure.num_residues}")
print(f"Confidence score:   {structure.confidence_score:.3f}")
print(f"Average pLDDT:      {structure.avg_plddt:.3f}")
print(f"pTM score:          {structure.ptm:.3f}")
print(f"ipTM score:         {structure.iptm:.3f}")

# Visualize the complex colored by chain to distinguish protein from ligand
structure.visualize(color_by="chain", style="ribbon")

## Export Results

Predicted structures returned by `predict_structures` can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD, and as input to further computational workflows. The B-factor column contains per-residue pLDDT confidence scores.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export the predicted structure to PDB
output.export(name="ruvb_adp_complex", export_path=output_dir, file_format="pdb")